# Illumina WGS age × sex counts (for manuscript Figure 1A)

Run in a **Verily Workbench** Jupyter app with CDR BigQuery access and the
`vwb-aou-datasets-controlled-v9` workspace mount.

**What it does**
1. Reads Illumina WGS `person_id`s from the v9 CRAM manifest (local file).
2. Queries CDR demographics read-only (`person` + `concept` for `sex_at_birth`).
3. Joins in pandas and writes an aggregated count table (no person ids):
   `age`, `sex`, `n`.

No BigQuery dataset create / table load is required (Workbench projects often
lack `bigquery.datasets.create`).

**Outputs** (under `~/illumina_age_sex/` by default)
- `illumina_wgs_age_sex_counts.tsv` — manuscript figure input
- `illumina_wgs_age_sex_summary.txt` — totals and small-cell notes

Age is capped at 90 (90+). Cells with `n < 20` are dropped from the export.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

# ---- edit these if needed ----
OUT_DIR = Path.home() / "illumina_age_sex"
MANIFEST_CANDIDATES = [
    Path.home() / "workspace" / "vwb-aou-datasets-controlled-v9" / "v9" / "wgs" / "cram" / "manifest.csv",
    Path("workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv"),
]
MIN_CELL = 20
AGE_CAP = 90
# Leave blank to use WORKSPACE_CDR from the Workbench snippet environment.
CDR_OVERRIDE = ""
# Optional hard override for age_at_cdr. Leave blank to try cdr_metadata, else CURRENT_DATE().
CDR_CUTOFF_DATE = ""  # e.g. "2024-10-01"

OUT_DIR.mkdir(parents=True, exist_ok=True)
COUNTS_TSV = OUT_DIR / "illumina_wgs_age_sex_counts.tsv"
SUMMARY_TXT = OUT_DIR / "illumina_wgs_age_sex_summary.txt"

print("OUT_DIR:", OUT_DIR)

## 1. Resolve CDR and CRAM manifest

In [ ]:
def resolve_cdr() -> str:
    if CDR_OVERRIDE.strip():
        return CDR_OVERRIDE.strip()
    # Same env var as the Workbench / AoU code snippets.
    value = (os.environ.get("WORKSPACE_CDR") or "").strip()
    if value:
        print(f"CDR from $WORKSPACE_CDR: {value}")
        return value
    raise RuntimeError("WORKSPACE_CDR is unset. Set CDR_OVERRIDE to project.dataset.")


def resolve_manifest() -> Path:
    for path in MANIFEST_CANDIDATES:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(
        "Could not find v9 WGS CRAM manifest. Expected one of:\n  "
        + "\n  ".join(str(p) for p in MANIFEST_CANDIDATES)
    )


CDR = resolve_cdr()
MANIFEST = resolve_manifest()
PROJECT = (
    os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("GOOGLE_PROJECT")
    or os.environ.get("GCP_PROJECT")
    or None
)
client = bigquery.Client(project=PROJECT)
print("CDR:", CDR)
print("PROJECT:", PROJECT or client.project)
print("MANIFEST:", MANIFEST)

## 2. Illumina WGS person ids from the CRAM manifest

In [ ]:
manifest = pd.read_csv(MANIFEST, dtype=str)
id_col = next(
    (c for c in ("person_id", "research_id", "sample_id") if c in manifest.columns),
    None,
)
if id_col is None:
    raise RuntimeError(f"manifest has no person id column; columns={list(manifest.columns)}")

wgs_ids = set(
    manifest[id_col]
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("") & s.str.lower().ne("nan")]
    .drop_duplicates()
)
print(f"Illumina WGS person ids: {len(wgs_ids):,}  (from {id_col})")

## 3. Resolve the CDR age cutoff date

In [ ]:
def table_exists(table: str) -> bool:
    sql = f"""
    SELECT 1
    FROM `{CDR}`.INFORMATION_SCHEMA.TABLES
    WHERE table_name = @table
    LIMIT 1
    """
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("table", "STRING", table)]
        ),
    )
    return next(job.result(), None) is not None


def columns_of(table: str) -> set[str]:
    sql = f"""
    SELECT column_name
    FROM `{CDR}`.INFORMATION_SCHEMA.COLUMNS
    WHERE table_name = @table
    """
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("table", "STRING", table)]
        ),
    )
    return {row.column_name for row in job.result()}


def resolve_cutoff_date() -> str:
    if CDR_CUTOFF_DATE.strip():
        print("CDR cutoff from CDR_CUTOFF_DATE:", CDR_CUTOFF_DATE.strip())
        return CDR_CUTOFF_DATE.strip()
    for table in ("cdr_metadata", "_cdr_metadata"):
        if not table_exists(table):
            continue
        cols = columns_of(table)
        for col in ("cdr_cutoff_date", "cutoff_date", "date", "value"):
            if col not in cols:
                continue
            sql = f"SELECT CAST({col} AS STRING) AS d FROM `{CDR}.{table}` LIMIT 5"
            try:
                rows = list(client.query(sql).result())
            except Exception as exc:  # noqa: BLE001
                print(f"skip {table}.{col}: {exc}")
                continue
            for row in rows:
                text = (row.d or "").strip()
                if len(text) >= 10 and text[4] == "-" and text[7] == "-":
                    print(f"CDR cutoff from {table}.{col}: {text[:10]}")
                    return text[:10]
    print("CDR cutoff not found in metadata; using CURRENT_DATE()")
    return ""


CUTOFF = resolve_cutoff_date()
AGE_DATE_SQL = f"DATE('{CUTOFF}')" if CUTOFF else "CURRENT_DATE()"

## 4. Pull demographics from CDR (read-only)

Uses the same `person` / `sex_at_birth_concept_id` join as the Workbench snippets.
Age is floor(day difference / 365.25), capped at 90, matching panel A.

In [ ]:
person_cols = columns_of("person")
if "sex_at_birth_concept_id" not in person_cols:
    raise RuntimeError(
        f"`{CDR}.person` has no sex_at_birth_concept_id; columns include: "
        + ", ".join(sorted(person_cols)[:40])
    )
if "birth_datetime" not in person_cols:
    raise RuntimeError(f"`{CDR}.person` has no birth_datetime")

sql = f"""
SELECT
  CAST(p.person_id AS STRING) AS person_id,
  LEAST(
    CAST(FLOOR(DATE_DIFF({AGE_DATE_SQL}, DATE(p.birth_datetime), DAY) / 365.25) AS INT64),
    {AGE_CAP}
  ) AS age,
  c.concept_name AS sex_at_birth
FROM `{CDR}.person` p
LEFT JOIN `{CDR}.concept` c
  ON c.concept_id = p.sex_at_birth_concept_id
"""
print(sql)
demo = client.query(sql).result().to_dataframe(create_bqstorage_client=False)
print(f"CDR person rows: {len(demo):,}")
print(demo["sex_at_birth"].value_counts(dropna=False).head(20))

## 5. Restrict to Illumina WGS, aggregate, suppress small cells

In [ ]:
demo["person_id"] = demo["person_id"].astype(str).str.strip()
wgs = demo[demo["person_id"].isin(wgs_ids)].copy()
print(f"WGS ids with CDR demographics: {len(wgs):,} / {len(wgs_ids):,}")

def normalize_sex(value) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).strip()
    low = text.lower()
    if low in {"female", "f"} or "female" in low:
        return "Female"
    if low in {"male", "m"} or ("male" in low and "female" not in low):
        return "Male"
    return text  # keep other labels for the summary


wgs["sex"] = wgs["sex_at_birth"].map(normalize_sex)
wgs = wgs[wgs["age"].notna() & (wgs["age"] >= 0)].copy()
wgs["age"] = wgs["age"].astype(int).clip(upper=AGE_CAP)

counts = (
    wgs.groupby(["age", "sex"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["age", "sex"])
)
print("Sex totals before small-cell suppression:")
print(counts.groupby("sex", dropna=False)["n"].sum().sort_values(ascending=False))

usable = counts[counts["sex"].isin(["Female", "Male"])].copy()
small = usable[usable["n"] < MIN_CELL].copy()
export = usable[usable["n"] >= MIN_CELL].copy()
export.to_csv(COUNTS_TSV, sep="\t", index=False)

other = counts[~counts["sex"].isin(["Female", "Male"])]
lines = [
    f"CDR={CDR}",
    f"manifest={MANIFEST}",
    f"cutoff={CUTOFF or 'CURRENT_DATE()'}",
    f"wgs_ids={len(wgs_ids)}",
    f"wgs_with_demographics={len(wgs)}",
    f"counted_female_male={int(usable['n'].sum())}",
    f"exported_after_min_cell_{MIN_CELL}={int(export['n'].sum())}",
    f"small_cells={len(small)} suppressed_n={int(small['n'].sum()) if len(small) else 0}",
]
if len(other):
    lines.append("non_female_male_sex_totals:")
    for sex, n in other.groupby("sex")["n"].sum().sort_values(ascending=False).items():
        lines.append(f"  {sex!r}: {int(n)}")
SUMMARY_TXT.write_text("\n".join(lines) + "\n")

print("\n".join(lines))
print("\nwrote", COUNTS_TSV)
print("wrote", SUMMARY_TXT)
display(export.head(12))

## 6. Download for the manuscript

Copy `~/illumina_age_sex/illumina_wgs_age_sex_counts.tsv` out of Workbench into the
`aou-lr-phase-2` repo (or send the path). Also note
`exported_after_min_cell_*` for the panel A caption.